# SetFit learning curve

Fine-tunes a biomedical sentence-transformer (S-PubMedBert-MS-MARCO) with SetFit on 8, 32, 128 and 512 labelled examples per class. T4 GPU, 20 to 35 minutes.

In [ ]:
# ===== STEP 3: fine-tune a biomedical sentence-transformer with SetFit (learning curve) =====
# Runtime > Change runtime type > T4 GPU before running.
!pip -q install setfit

import os, time, json
os.environ["WANDB_DISABLED"] = "true"      # stops the wandb sign-in prompt
os.environ["WANDB_MODE"] = "disabled"
import numpy as np, pandas as pd, torch
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset
from sklearn.metrics import f1_score, accuracy_score
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/sci-classify/data"
RES = "/content/drive/MyDrive/sci-classify/results"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to a T4 runtime")

train = pd.read_parquet(f"{DATA}/train.parquet")
test = pd.read_parquet(f"{DATA}/test.parquet")
test_llm = pd.read_parquet(f"{DATA}/test_llm_1000.parquet")
LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]

# A sentence-transformer pre-trained on biomedical text (PubMedBERT). Falls back to a small general model if unavailable.
MODEL_IDS = ["pritamdeka/S-PubMedBert-MS-MARCO", "sentence-transformers/all-MiniLM-L6-v2"]

def with_position(df):
    """Give the model the sentence's position as text, e.g. '[3 of 9] ...', the same clue the baseline used."""
    return [f"[{l + 1} of {t}] {s}" for s, l, t in zip(df.text, df.line_number, df.total_lines)]

def bootstrap_ci(y, p, n=1000, seed=0):
    rng = np.random.default_rng(seed); y, p = np.asarray(y), np.asarray(p); idx = np.arange(len(y))
    s = [f1_score(y[i], p[i], average="macro") for i in (rng.choice(idx, len(idx)) for _ in range(n))]
    return np.percentile(s, [2.5, 97.5])

pool = Dataset.from_pandas(pd.DataFrame({"text": train.text, "text_pos": with_position(train), "label": train.label}))

def run(n_per_class, seed, use_position=False, full_test=False):
    sample = sample_dataset(pool, label_column="label", num_samples=n_per_class, seed=seed)
    text_col = "text_pos" if use_position else "text"
    ds = Dataset.from_dict({"text": sample[text_col], "label": sample["label"]})
    for mid in MODEL_IDS:
        try:
            model = SetFitModel.from_pretrained(mid, labels=LABELS); used = mid; break
        except Exception as e:
            print("Could not load", mid, "->", e)
    iters = 20 if n_per_class <= 128 else 5          # fewer contrastive pairs per example at larger sizes keeps runtime short
    args = TrainingArguments(batch_size=32, num_epochs=1, num_iterations=iters, seed=seed, report_to="none")
    t0 = time.time(); Trainer(model=model, args=args, train_dataset=ds).train(); secs = time.time() - t0
    x_llm = with_position(test_llm) if use_position else test_llm.text.tolist()
    p_llm = list(model.predict(x_llm))
    out = {"model": f"SetFit ({used.split('/')[-1]})" + (" + position" if use_position else ""),
           "n_per_class": n_per_class, "seed": seed, "train_seconds": round(secs),
           "macro_f1_1000": f1_score(test_llm.label, p_llm, average="macro"),
           "acc_1000": accuracy_score(test_llm.label, p_llm), "preds_1000": p_llm}
    if full_test:
        x_full = with_position(test) if use_position else test.text.tolist()
        t1 = time.time(); p_full = list(model.predict(x_full, batch_size=256)); pred_secs = time.time() - t1
        lo, hi = bootstrap_ci(test.label, p_full)
        out.update({"macro_f1_full": f1_score(test.label, p_full, average="macro"), "full_ci": (lo, hi),
                    "ms_per_sentence": 1000 * pred_secs / len(test)})
    print(f"{out['model']:45s} n={n_per_class:>3}/class seed={seed}  macro-F1 (1000 set) {out['macro_f1_1000']:.3f}"
          + (f"  | full test {out['macro_f1_full']:.3f} ({out['full_ci'][0]:.3f} to {out['full_ci'][1]:.3f}),"
             f" {out['ms_per_sentence']:.2f} ms/sentence" if full_test else "") + f"  [{secs:.0f}s training]")
    return out

results = []
# Learning curve, text only. Small sizes are noisy, so they get 3 seeds each.
for n, seeds in [(8, [0, 1, 2]), (32, [0, 1, 2]), (128, [0]), (512, [0])]:
    for s in seeds:
        results.append(run(n, s, full_test=(n == 512)))
# Same as the best setting, but with the position clue, for a fair comparison with the position baseline
results.append(run(512, 0, use_position=True, full_test=True))

summary = (pd.DataFrame([{k: v for k, v in r.items() if k not in ("preds_1000",)} for r in results])
             .groupby(["model", "n_per_class"]).macro_f1_1000.agg(["mean", "std", "count"]).round(3))
print("\nLearning curve (macro-F1 on the 1,000-sentence set):"); print(summary.to_string())

json.dump([{k: (list(v) if isinstance(v, tuple) else v) for k, v in r.items()} for r in results],
          open(f"{RES}/step3_setfit.json", "w"))
print("\nSaved to", RES)
